# 04 - Silver Past Calendar Rates

## Objetivo

Realizar a transformação e padronização da tabela
`past_calendar_rates`, proveniente da camada Bronze,
preparando os dados históricos de desempenho dos anúncios
para análises do mercado de Airbnb em João Pessoa - PB.

## Fonte

`airbnb_joao_pessoa.bronze.past_calendar_rates`

## Granularidade

Cada registro representa um anúncio (`listing_id`) em uma
determinada data (`date`).

Chave lógica:

`listing_id + date`

## Responsabilidades

- avaliação da qualidade dos dados;
- verificação de duplicidades;
- tratamento de valores nulos;
- padronização de tipos;
- validação de datas;
- validação dos domínios numéricos;
- criação de indicadores de disponibilidade;
- organização das colunas;
- persistência na camada Silver.

## Tabela de saída

`airbnb_joao_pessoa.silver.past_calendar_rates`

In [0]:
# Leitura
past_rates = spark.table("airbnb_joao_pessoa.bronze.past_calendar_rates")

display(past_rates)

In [0]:
print(f"Quantidade de registros: {past_rates.count()}")
print(f"Quantidade de colunas: {len(past_rates.columns)}")

In [0]:
past_rates.printSchema()

In [0]:
# Analise dos nulos
from pyspark.sql.functions import sum, when, col

null_summary = (past_rates.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in past_rates.columns]))

display(null_summary)

In [0]:
# Analise de duplicados
duplicated_rates = (past_rates.groupBy("listing_id", "date").count().filter(col("count") > 1))

display(duplicated_rates)

In [0]:
# Datas

from pyspark.sql.functions import min, max

display(past_rates.select(min("date").alias("min_date"), max("date").alias("max_date")))

In [0]:
from pyspark.sql.functions import dayofmonth

display(past_rates.filter(dayofmonth("date") != 1).select("listing_id", "date"))